In [ ]:
from email.message import EmailMessage
import os
import smtplib
import requests
from openai.types.responses import ResponseTextDeltaEvent
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel, set_tracing_disabled
from IPython.display import Markdown, display
import asyncio

set_tracing_disabled(disabled=True)

ollama_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
local_llm = OpenAIChatCompletionsModel(model="qwen3:8b", openai_client=ollama_client)

smtp_server = os.getenv("EMAIL_SMTP_SERVER")
mail_app_password = os.getenv("EMAIL_APP_PASSWORD")
email_address = os.getenv("EMAIL_ADDRESS")
USE_EMAIL = True

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = email_address
    msg["To"] = email_address
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(smtp_server, 587) as server:
        server.starttls()
        server.login(email_address, mail_app_password)
        server.send_message(msg)

@function_tool
def send_message(subject, text_body, html_body):
    """ Send the message either to the email or print based on the USE_EMAIL variable """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        print(f"Subject: {subject}\n\n{text_body}")

@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_email(subject, text_body, html_body)
    return "Email sent successfully"


In [2]:
intro= """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write short emails.
"""

professional_email_instructions = intro + "Your email style is professional, serious, with gravities and credibility."
humorous_email_instruction = intro + "Your email is witty, engaging and humorous."
executive_email_instructions = intro + "Your email is concise, to the point, in the style of a busy senior executive."

professional_agent = Agent(name="Professional Sales Agent", model=local_llm, instructions=professional_email_instructions)
humorous_agent = Agent(name="Humorous Sales Agent", model=local_llm, instructions=humorous_email_instruction)
executive_agent = Agent(name="Executive Sales Agent", model=local_llm, instructions=executive_email_instructions)

In [3]:
from agents import ModelSettings, model_settings


decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""
require_tool = ModelSettings(tool_choice="required")
sales_sender = Agent(name="Sales Sender", instructions=decision, model=local_llm, tools=[send_email_tool], model_settings=require_tool)

In [4]:
message = "Write a cold sales email"

results = await asyncio.gather(
    Runner.run(professional_agent, message),
    Runner.run(humorous_agent, message),
    Runner.run(executive_agent, message)
)

outputs = [result.final_output for result in results]

emails = f"Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

best = await Runner.run(sales_sender, emails)

print(f"Final response:\n{best.final_output}")

Final response:
{"name": "send_email_tool", "arguments": {"subject": "Simplify Your SOC2 Compliance with the Help of AI!", "text_body": "I hope this email finds you well! 😊 We at ComplAI understand that navigating through SOC2 compliance can be as tricky as deciphering hieroglyphics with your arms tied behind your back. 💪 Our SaaS tool, powered by the magic of AI, is designed to make SOC2 preparation a breeze. Imagine waking up every morning feeling refreshed and ready for whatever come your way without having to pour over a mile-long checklist every time!\r\nWhy settle for boring SOC2 compliance when you can have fun and avoid the headache? 🎉 Whether you're dealing with annual audits or just need that extra bit of comfort knowing everything is in order, our tool has got your back. And best of all? With AI doing all the heavy lifting, all you need to do is sit back and relax.\r\nDon't let compliance be a drain on your spirit—let ComplAI energize it for you! 🌟 Let’s take those mundane t